In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df=pd.read_csv("pipeline_data_id.csv")

In [3]:
def get_preprocessor():
    categorical = ['since_last_login']
    ordinal = ['company_size']
    binary = ['is_premium', 'is_unlimited', 'subscription_status']
    numeric = ['sub_user', 'subscription_count', 'invitations', 'pre_recorded_kit_counts', 'jobs_posted', 'pre_rec_interview_taken']
    log_scaled = ['subscription_sum', 'wallet_amount']

    preprocessor = ColumnTransformer([
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical),
        ('ord', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(categories=[['1-25', '26-100', '101-500', '500-1000', 'More than 1000']],
                                       handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ordinal),
        ('bin', SimpleImputer(strategy='mean'), binary),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scale', StandardScaler())
        ]), numeric),
        ('log', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('log', FunctionTransformer(np.log1p, validate=False)),
            ('scale', StandardScaler())
        ]), log_scaled)
    ])
    return preprocessor

def preprocess_data(df, preprocessor=None, fit=True):
    if fit or preprocessor is None:
        preprocessor = get_preprocessor()
        features = preprocessor.fit_transform(df)
    else:
        features = preprocessor.transform(df)
    return features, preprocessor

# --- Autoencoder Model ---
class ClientAutoencoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

def to_tensor(X):
    return torch.FloatTensor(X.toarray() if hasattr(X, 'toarray') else X)

def train_autoencoder(X_train, model_path, input_dim, epochs=50, batch_size=32):
    X_tensor = to_tensor(X_train)
    dataset = TensorDataset(X_tensor, X_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ClientAutoencoder(input_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in dataloader:
            inputs, _ = batch
            inputs = inputs.to(device)

            optimizer.zero_grad()
            _, outputs = model(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

    torch.save(model.state_dict(), model_path)
    return model

# --- Model Loading ---
def load_or_train_autoencoder(X, model_path='client_autoencoder.pth'):
    input_dim = X.shape[1]
    model = ClientAutoencoder(input_dim)
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location='cpu'))
        print("Model loaded from disk.")
    else:
        print("No model found. Training new model...")
        model = train_autoencoder(X, model_path, input_dim)
    return model

# --- Embedding Extraction ---
def extract_embeddings(model, X):
    model.eval()
    with torch.no_grad():
        tensor = to_tensor(X)
        embeddings, _ = model(tensor)
    return embeddings.cpu().numpy()

# --- Recommendations ---
def generate_recommendations(embeddings, client_ids, top_k=5, min_similarity=0.7):
    similarity_matrix = cosine_similarity(embeddings)
    recommendations = {}
    for idx, client_id in enumerate(client_ids):
        sim_scores = similarity_matrix[idx]
        top_indices = np.argsort(sim_scores)[::-1]
        similar_clients = []
        for i in top_indices:
            if client_ids[i] == client_id:
                continue
            if sim_scores[i] >= min_similarity:
                similar_clients.append((client_ids[i], round(sim_scores[i], 4)))
            if len(similar_clients) >= top_k:
                break
        recommendations[client_id] = similar_clients
    return recommendations

# --- Entry Point ---
def get_similar_clients(input_client_id, df, top_k=5, min_similarity=0.7):
    client_ids = df['jobma_catcher_id'].values
    df_cleaned = df.drop('jobma_catcher_id', axis=1)

    features, preprocessor = preprocess_data(df_cleaned, fit=True)
    model = load_or_train_autoencoder(features)
    embeddings = extract_embeddings(model, features)
    recs = generate_recommendations(embeddings, client_ids, top_k, min_similarity)

    if input_client_id not in recs:
        print(f"Client ID {input_client_id} not in dataset.")
        return pd.DataFrame()

    similar_ids = [cid for cid, _ in recs[input_client_id]]
    scores = [score for _, score in recs[input_client_id]]
    result = df[df['jobma_catcher_id'].isin(similar_ids)].copy()
    result['similarity_score'] = scores
    return result.reset_index(drop=True)

if __name__ == "__main__":
    df = pd.read_csv("pipeline_data_id.csv")
    input_id = 10521  # change this as needed
    recommendations_df = get_similar_clients(input_id, df)
recommendations_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,invitations,pre_recorded_kit_counts,jobs_posted,pre_rec_interview_taken,since_last_login,similarity_score
0,8535,1,1,1-25,0,14746.0,0,197.8,8,0,1,19,0,4,0.9054
1,9891,0,1,26-100,1,500000.0,1,117.6,1,81,19,18,48,4,0.8145
2,9994,0,1,1-25,2,499998.0,1,0.0,1,255,5,24,50,3,0.7810
3,10056,0,1,26-100,2,499970.0,1,117.6,1,276,43,30,66,3,0.7686
4,10259,0,1,1-25,1,898.0,0,1.2,1,17,6,10,9,3,0.7472


In [5]:
if __name__ == "__main__":
    df = pd.read_csv("pipeline_data_id.csv")
    input_id = 10259
    recommendations_df = get_similar_clients(input_id, df)
recommendations_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,invitations,pre_recorded_kit_counts,jobs_posted,pre_rec_interview_taken,since_last_login,similarity_score
0,9969,0,1,1-25,1,188.0,0,0.6,1,33,6,9,17,4,0.8966
1,9994,0,1,1-25,2,499998.0,1,0.0,1,255,5,24,50,3,0.8670
2,10397,0,1,1-25,1,772.0,0,1.2,1,147,20,23,87,3,0.8652
3,10496,0,1,26-100,1,80.0,0,1.2,1,53,8,7,22,3,0.8610
4,10499,0,1,1-25,1,161.0,0,3.0,2,24,2,2,9,3,0.8441


In [6]:
if __name__ == "__main__":
    df = pd.read_csv("pipeline_data_id.csv")
    input_id = 9969
    recommendations_df = get_similar_clients(input_id, df)
recommendations_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,invitations,pre_recorded_kit_counts,jobs_posted,pre_rec_interview_taken,since_last_login,similarity_score
0,9045,0,1,1-25,1,60.0,0,0.6,1,18,4,7,7,4,0.9476
1,9931,0,1,26-100,1,296.0,0,0.6,1,32,6,6,8,4,0.9248
2,10095,0,1,1-25,0,117.0,0,0.0,1,20,11,9,3,4,0.9002
3,10155,0,1,26-100,1,195.0,0,0.6,1,20,4,10,11,4,0.8978
4,10211,0,1,1-25,1,49.0,0,0.0,1,0,1,7,0,4,0.8840
